# 03 - Train

Notebook này huấn luyện 4 model hiện có. Tuning dùng CV trên train set, không dùng test set để tune.

## Bước 3: Huấn luyện và Đánh giá Mô hình

Chúng ta sẽ xây dựng và so sánh 4 mô hình hồi quy với các cơ chế hoạt động khác nhau. Để đảm bảo đo lường dung lượng file mô hình thực tế, chúng ta sẽ viết một hàm helper để ghi lại toàn bộ các chỉ số hiệu năng bao gồm thời gian chạy và kích thước file `.pkl` của từng mô hình.

In [ ]:
from pathlib import Path
import sys

from IPython.display import display

sys.path.append(str(Path('../src').resolve()))
from train import train_models

RUN_HYPERPARAMETER_SEARCH = True
result = train_models(Path('../data/housing.csv.zip'), run_search=RUN_HYPERPARAMETER_SEARCH)

display(result['comparison'])
print('Best model:', result['best_model_name'])

### Giải thích Chi tiết các Mô hình & Phân tích Đánh đổi (Trade-off)

#### 1. Lý do chọn các dải siêu tham số & Ý nghĩa siêu tham số tối ưu thu được
- **Decision Tree (`max_depth`, `min_samples_split`):** Chọn độ sâu từ 10 đến không giới hạn nhằm tìm điểm cân bằng giúp cây có thể biểu diễn tốt quan hệ phi tuyến mà không phân chia quá sâu dẫn đến học vẹt (overfitting). `min_samples_split` giới hạn số lượng mẫu cần thiết ở một nút để tiến hành phân nhánh nhằm giảm thiểu việc tạo các nhánh lá quá nhỏ.
- **Random Forest (`n_estimators`, `max_depth`, `min_samples_split`):** Số lượng cây được lựa chọn từ 50 đến 150 để cân nhắc hiệu năng thời gian tính toán và độ mượt của đường biên dự đoán. Do cơ chế Bagging trung bình cộng kết quả của nhiều cây độc lập, dải `max_depth` lớn hơn vẫn giữ được tính ổn định và ít bị overfitting hơn cây đơn lẻ.
- **Gradient Boosting (`n_estimators`, `learning_rate`, `max_depth`):** Do đặc trưng học tuần tự và sửa sai, chúng ta chọn độ sâu cây nhỏ hơn (3-7) để mỗi cây chỉ hoạt động như một weak learner. `learning_rate` điều khiển kích thước bước điều chỉnh sau mỗi cây lỗi nhằm hội tụ mượt mà và tránh nhảy vọt qua điểm tối ưu.

#### 2. Cơ chế hoạt động & Ưu/Nhược điểm của các họ mô hình
- **Linear Regression (Baseline):**
  - *Cơ chế:* Cố gắng khớp các hệ số trọng số tuyến tính cho từng đặc trưng thông qua tối thiểu hóa bình phương sai số (OLS).
  - *Ưu điểm:* Cực kỳ nhanh, dễ giải thích ý nghĩa hệ số trực quan.
  - *Nhược điểm:* Giả định quan hệ tuyến tính nghiêm ngặt, kém hiệu quả với dữ liệu phi tuyến phức tạp như phân bố địa lý của giá nhà.
- **Decision Tree:**
  - *Cơ chế:* Phân chia không gian dữ liệu bằng các ngưỡng điều kiện dọc theo các đặc trưng số để tối đa độ thuần nhất ở mỗi vùng.
  - *Ưu điểm:* Không đòi hỏi chuẩn hóa cao, bắt được quan hệ phi tuyến và tương tác đặc trưng nguyên bản.
  - *Nhược điểm:* Rất dễ bị biến động mạnh khi dữ liệu thay đổi nhỏ (High Variance) và cực kỳ dễ Overfitting.
- **Random Forest (Ensemble - Bagging):**
  - *Cơ chế:* Huấn luyện song song nhiều cây quyết định độc lập trên các mẫu bootstrap khác nhau, lấy ngẫu nhiên tập con đặc trưng (Feature Subspacing) để tăng tính đa dạng, sau đó lấy trung bình dự đoán.
  - *Ưu điểm:* Cực kỳ bền bỉ trước overfitting, độ chính xác cao.
  - *Nhược điểm:* File lưu trữ có kích thước rất lớn do phải chứa cấu trúc của hàng trăm cây, thời gian dự đoán chậm hơn do cần tổng hợp từ tất cả các cây.
- **Gradient Boosting (Ensemble - Boosting):**
  - *Cơ chế:* Huấn luyện tuần tự các cây quyết định ngắn, cây sau cố gắng khớp phần dư thừa (residual errors) mà các cây trước đó dự đoán sai lệch.
  - *Ưu điểm:* Độ chính xác cực kỳ cao, thường là quán quân của các mô hình học máy truyền thống.
  - *Nhược điểm:* Huấn luyện chậm do tính tuần tự (không song song hóa huấn luyện hoàn toàn như Bagging), nhạy cảm với các siêu tham số.

#### 3. Phân tích sự Đánh đổi (Trade-off) khi Triển khai (Deployment)
- **Độ chính xác (RMSE) vs Thời gian dự đoán:** Linear Regression dự đoán ngay lập tức (Inference Time gần như bằng 0) nhưng RMSE lại rất cao. Gradient Boosting hoặc Random Forest cho RMSE tốt nhất nhưng phải duyệt qua toàn bộ cấu trúc cây phức tạp khiến thời gian suy diễn tăng lên. Đối với ứng dụng Real-time cần phản hồi dưới vài mili-giây, Linear Regression hoặc một cây quyết định nông sẽ được ưu tiên, còn đối với các tác vụ chạy theo lô (Batch Processing) đề cao độ chính xác, Gradient Boosting là lựa chọn tối ưu.
- **Kích thước file mô hình:** Random Forest có kích thước file nặng nhất (thường hàng chục đến hàng trăm MB tùy độ sâu và số lượng đặc trưng) do chứa thông tin phân nhánh của rất nhiều cây phức tạp độc lập, gây khó khăn cho việc đóng gói vào các dịch vụ Serverless siêu nhẹ (như AWS Lambda) do giới hạn bộ nhớ đệm. Ngược lại, Linear Regression hay Gradient Boosting (nhờ cấu trúc cây nông hơn) giữ được dung lượng lưu trữ gọn gàng và dễ phân phối hơn.